<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/training/train_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Inisialisasi dan Akses Data

In [2]:
from google.colab import drive
import pandas as pd
import os

# Menghubungkan ke Google Drive
drive.mount('/content/drive')

# Mengatur path sesuai struktur folder ganda yang terlihat di sidebar
path_raw = '/content/drive/MyDrive/Data/Data/Raw/'
path_processed = '/content/drive/MyDrive/Data/Data/Processed/'

# Verifikasi akses folder
if os.path.exists(path_raw):
    print("Koneksi sukses. File di folder Raw:")
    print(os.listdir(path_raw))
else:
    print("Path salah, silakan cek kembali folder di Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Koneksi sukses. File di folder Raw:
['Scales Reference.xlsx', 'Job Zone Reference.xlsx', 'Job Zones.xlsx', 'Occupation Data.xlsx', 'Skills.xlsx', 'Task Statements.xlsx', 'Work Context.xlsx', 'Work Activities.xlsx']


## 2. Memuat Dataset Utama

In [3]:
# Membaca data pekerjaan dan tugas dari file Excel
df_occupation = pd.read_excel(path_raw + 'Occupation Data.xlsx')
df_tasks = pd.read_excel(path_raw + 'Task Statements.xlsx')

# Memuat data master (asumsi file ini berformat .csv di folder Processed)
df_master = pd.read_csv(path_processed + 'occupation_master_final.csv')

print("Data sukses dimuat. Jumlah baris pekerjaan:", len(df_occupation))

Data sukses dimuat. Jumlah baris pekerjaan: 1016


## 3. Penggabungan Teks (Preprocessing)

In [4]:
# Mengelompokkan tugas berdasarkan kode pekerjaan
tasks_joined = df_tasks.groupby('O*NET-SOC Code')['Task'].apply(lambda x: ' '.join(x.astype(str))).reset_index()

# Menggabungkan data utama dengan tugas
df_merged = pd.merge(df_occupation, tasks_joined, on='O*NET-SOC Code', how='left')

# Membuat kolom teks gabungan (Judul + Deskripsi + Tugas)
df_merged['text_full'] = df_merged['Title'].fillna('') + " " + \
                         df_merged['Description'].fillna('') + " " + \
                         df_merged['Task'].fillna('')

print("Proses penggabungan teks selesai")

Proses penggabungan teks selesai


## 4. Ekstraksi Fitur dengan TF-IDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF dengan membatasi 500 kata paling penting agar tidak terlalu berat
tfidf = TfidfVectorizer(max_features=500, stop_words='english')

# Mengubah teks pada kolom 'text_full' menjadi matriks angka
tfidf_matrix = tfidf.fit_transform(df_merged['text_full'])

# Menyimpan hasil ekstraksi ke dalam DataFrame baru
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

print("Fitur teks berhasil diekstrak")
df_tfidf.head()

Fitur teks berhasil diekstrak


,abreast,academic,according,accounts,accuracy,activities,address,adjust,adjustments,administer,...,water,web,wind,wood,work,workers,working,workpieces,write,written
0,0.000000,0.0,0.0,0.000000,0.0,0.274514,0.000000,0.0,0.0,0.044416,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.041353,0.0
1,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.149415,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.077663,0.0,0.121718,0.0
2,0.000000,0.0,0.0,0.000000,0.0,0.154055,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.070138,0.000000,0.000000,0.0,0.000000,0.0
3,0.068108,0.0,0.0,0.080717,0.0,0.038995,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.047085,0.067466,0.0,0.052868,0.0
4,0.000000,0.0,0.0,0.100679,0.0,0.024319,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.058729,0.042075,0.0,0.000000,0.0


In [6]:
# Mencari nilai tertinggi di matriks untuk memastikan data terisi
print("Nilai maksimal di seluruh matriks:", df_tfidf.max().max())

# Melihat kata-kata yang memiliki bobot tertinggi
print("\n10 Kata dengan bobot tertinggi secara keseluruhan:")
print(df_tfidf.max().sort_values(ascending=False).head(10))

Nilai maksimal di seluruh matriks: 1.0

10 Kata dengan bobot tertinggi secara keseluruhan:
teachers        1.0
teaching        1.0
construction    1.0
specialists     1.0
computer        1.0
managers        1.0
life            1.0
financial       1.0
engineers       1.0
equipment       1.0
dtype: float64


## 5. Penggabungan Fitur Teks dan Data Tabular

In [7]:
# Menyatukan data master dengan fitur TF-IDF secara berdampingan (kolom)
df_final_train = pd.concat([df_master, df_tfidf], axis=1)

# Menampilkan informasi hasil penggabungan
print("Data siap digunakan untuk proses training")
print("Total fitur sekarang:", df_final_train.shape[1])
df_final_train.head()

Data siap digunakan untuk proses training
Total fitur sekarang: 573


,occupation_code,title,description,skill_count,skill_importance_avg,skill_level_avg,top_skills,task_count,core_task_count,supplemental_task_count,...,water,web,wind,wood,work,workers,working,workpieces,write,written
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,35.0,3.188571,3.365143,"Judgment and Decision Making, Critical Thinkin...",31.0,20.0,11.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.041353,0.0
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",35.0,2.918857,2.821143,"Writing, Critical Thinking, Reading Comprehens...",18.0,18.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.077663,0.0,0.121718,0.0
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",35.0,2.852857,2.724286,"Reading Comprehension, Active Listening, Speak...",17.0,9.0,8.0,...,0.0,0.0,0.0,0.0,0.070138,0.000000,0.000000,0.0,0.000000,0.0
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes...",35.0,2.624286,2.421429,unknown,30.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.047085,0.067466,0.0,0.052868,0.0
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",35.0,2.709429,2.652571,"Active Listening, Speaking, Critical Thinking,...",30.0,13.0,8.0,...,0.0,0.0,0.0,0.0,0.000000,0.058729,0.042075,0.0,0.000000,0.0


## 6: Ekspor Data Siap Latih

In [8]:
# Menyimpan hasil akhir ke folder Processed dalam format CSV
output_path = path_processed + 'data_siap_training.csv'
df_final_train.to_csv(output_path, index=False)

print("File data_siap_training.csv telah berhasil disimpan di Drive")

File data_siap_training.csv telah berhasil disimpan di Drive
